In [2]:
# Install starter package
!pip install -q git+https://github.com/zhouy185/rosa-starter.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

In [97]:
times = delivery_times("Far West", "Fri/Sat eve", 45, seed=42)
print(len(times), "orders; average delivery time", round(times.mean(), 1),
"minutes")

210 orders; average delivery time 43.0 minutes


# Part I. Describe

## a. Percentage of late orders

In [15]:
def late_orders_pct(zone, time_block, promise):
  if zone != 'all' and time_block != 'all':
    pairs = [(zone, time_block)]
  elif zone != 'all' and time_block == "all":
    pairs = [(zone, item) for item in TIME_BLOCKS]
  elif time_block != 'all' and zone == "all":
    pairs = [(item, time_block) for item in ZONES]
  else:
    pairs = [(all_zone, all_time)
               for all_zone in ZONES for all_time in TIME_BLOCKS]

  late_orders = 0
  all_orders = 0
  for z, t in pairs:
    all_times = delivery_times(z, t, promise, seed=42)
    late_orders += np.sum(all_times > promise)
    all_orders += len(all_times)

  return round(late_orders / all_orders * 100, 2)


np.float64(32.86)

In [ ]:
# late_orders_pct("Far West", "Fri/Sat eve", 45)
# late_orders_pct("all", "Fri/Sat eve", 45)
# late_orders_pct("Far West", "all", 45)
# late_orders_pct("all", "all", 45)

## b. Rank pairs of zone and time blocks in descending order of late arrival rate

In [61]:
late_rates = {(z, t): late_orders_pct(z, t, 45) for z in ZONES for t in TIME_BLOCKS}

late_ranking = sorted(late_rates.items(), key=lambda pair: pair[1], reverse=True)

for rank, ((z, t), rate) in enumerate(late_ranking, start=1):
    print(f"{rank}. {z}, {t}: {rate}%")

1. Far West, Fri/Sat eve: 48.57%
2. North, Fri/Sat eve: 24.67%
3. Far West, Weekday eve: 17.06%
4. Central, Fri/Sat eve: 11.58%
5. North, Weekday eve: 11.2%
6. Far West, Other: 7.81%
7. Central, Weekday eve: 6.67%
8. Far West, Lunch: 5.83%
9. North, Other: 4.38%
10. Central, Lunch: 4.23%
11. Central, Other: 3.68%
12. North, Lunch: 3.68%


## c. Average delivery time

In [98]:
def avg_time(zone, time_block, promise):
  if zone != 'all' and time_block != 'all':
    pairs = [(zone, time_block)]
  elif zone != 'all' and time_block == "all":
    pairs = [(zone, item) for item in TIME_BLOCKS]
  elif time_block != 'all' and zone == "all":
    pairs = [(item, time_block) for item in ZONES]
  else:
    pairs = [(all_zone, all_time)
               for all_zone in ZONES for all_time in TIME_BLOCKS]

  total_minutes = 0
  all_orders = 0
  for z, t in pairs:
    all_times = delivery_times(z, t, promise, seed=42)
    total_minutes += sum(np.sum(times) for times in all_times)
    all_orders += len(all_times)

  return round(total_minutes / all_orders, 2)

# avg_time("Far West", "Fri/Sat eve", 45)
# avg_time("all", "Fri/Sat eve", 45)
# avg_time("Far West", "all", 45)
# avg_time("all", "all", 45)


## d. Rank pairs of zone and time blocks in descending order of average delivery time

In [72]:
avg_times = {(z, t): avg_time(z, t, 45) for z in ZONES for t in TIME_BLOCKS}

avg_ranking = sorted(avg_times.items(), key=lambda pair: pair[1], reverse=True)

for rank, ((z, t), avg) in enumerate(avg_ranking, start=1):
    print(f"{rank}. {z}, {t}: {avg} mins")

1. Far West, Fri/Sat eve: 41.55 mins
2. North, Fri/Sat eve: 37.33 mins
3. Far West, Weekday eve: 35.65 mins
4. North, Weekday eve: 32.81 mins
5. Central, Fri/Sat eve: 32.52 mins
6. Central, Weekday eve: 31.25 mins
7. Far West, Lunch: 30.8 mins
8. Far West, Other: 30.55 mins
9. North, Lunch: 30.48 mins
10. North, Other: 28.99 mins
11. Central, Lunch: 27.12 mins
12. Central, Other: 26.47 mins


## e. Comparing the relevance of late delivery rate vs average delivery time

Looking at the results from part b) and ), the late delivery rate would be more relevant to Rosa decisions.

The average delivery time is not as relevant since it doesn't show how often the 45-minute is broken. Also, from a client's perspective, if they live in the Far West and order on a Friday evening, their delivery time can be up to 50 minutes for multiple times, while on average, the delivery time is around 42-45 minutes. Thus, the average delivery time cannot reflect exactly the extent of the promise being broken. It is still, however, useful to illustrate the typical speed and overall efficiency of all deliveries.

The late delivery rate is more relevant because it measures the customer's experience of the promise. With the same example of the Far west region, on a Friday or Saturday evening: the average delivery time is in the range of low-to-mid 40 minutes. Nevertheless, the late delivery rate is around 45%, which means almost half of the deliveries were not on time. Hence, the late delivery rate would be way more useful for Rosa to reconsider her resource allocation or adjust her promise.

# Part II. Exploring best promise

## a. Total cost per late order

In [75]:
print(COSTS)

{'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}


In [80]:
def total_cost_per_late_order(costs):
  refund = costs['refund']
  lost_profit = costs['churn_orders'] * costs['margin']
  return round(refund + lost_profit, 2)

In [87]:
print("Total cost per late order: $", total_cost_per_late_order(COSTS))

Total cost per late order: $ 26.2


## b. Best promised delivery time

### b. i. Choosing a range of duration to test

The duration range should start at the shortest delivery time so that almost every order would be late, and end at the longest delivery time so that almost none would be late. The best promised time will always be within this range.

Based on the last four weeks:

In [100]:
all_times = np.concatenate([delivery_times(z, t, 45, seed=42)
                          for z in ZONES for t in TIME_BLOCKS])
shortest_time = all_times.min()
longest_time = all_times.max()
print("Shortest delivery:", shortest_time, "min")
print("Longest delivery: ", longest_time, "min")

Shortest delivery: 13.4 min
Longest delivery:  100.7 min


**Note:** If the best promised time lies at the shortest or longest delivery time, we have to widen this range. This is because it's likely we will be to find a better promised time outside this range

### b. ii. Best promise for each pair of zone and time block

The best promised time will be the time with the highest profit.

In [119]:
def net_profit(zone, time_block, promise, costs):
  # Net profit is the margin earned on all orders
  # minus the cost of the late ones
  times = delivery_times(zone, time_block, promise, seed=42)
  revenue = len(times) * costs['margin']
  late_cost = np.sum(times > promise) * total_cost_per_late_order(costs)
  return revenue - late_cost

def best_promise(zone, time_block, promises, costs):
  profits = {p: net_profit(zone, time_block, p, costs) for p in promises}
  best = max(profits, key=profits.get)
  return round(best, 2), round(profits[best], 2)

In [ ]:
promises = list(np.arange(shortest_time, longest_time + 2.5, 5))

# Testing for Far West region on Fri/Sat evening
best, profit = best_promise("Far West", "Fri/Sat eve", promises, COSTS)
print(f"Far West, Fri/Sat eve -> best promise: {best} min, net profit: ${profit}")

Far West, Fri/Sat eve -> best promise: 53.4 min, net profit: $1171.4


In [130]:
# Testing best promised time for all regions

for z in ZONES:
  for t in TIME_BLOCKS:
    best, profit = best_promise(z, t, promises, COSTS)
    print(f"{z}, {t}: Best promised time is: {best} mins - generating ${profit}")

Central, Lunch: Best promised time is: 48.4 mins - generating $2082.2
Central, Weekday eve: Best promised time is: 48.4 mins - generating $2510.0
Central, Fri/Sat eve: Best promised time is: 48.4 mins - generating $1418.6
Central, Other: Best promised time is: 43.4 mins - generating $1597.8
North, Lunch: Best promised time is: 48.4 mins - generating $1497.2
North, Weekday eve: Best promised time is: 48.4 mins - generating $1817.8
North, Fri/Sat eve: Best promised time is: 53.4 mins - generating $1027.6
North, Other: Best promised time is: 43.4 mins - generating $1292.6
Far West, Lunch: Best promised time is: 48.4 mins - generating $981.8
Far West, Weekday eve: Best promised time is: 53.4 mins - generating $1197.8
Far West, Fri/Sat eve: Best promised time is: 53.4 mins - generating $1171.4
Far West, Other: Best promised time is: 48.4 mins - generating $1480.0


# Part III. Creating a Web app

- GitHub Repo: https://github.com/h4hoa/Rosa-pizza-delivery
- Streamlit app link: https://rosa-pizza-delivery-hanna-nguyen.streamlit.app/

